[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramonzaca/MLSecOPs/blob/main/TP_04/04_bias_and_explainability_v3.ipynb)

**What's the model doing? - Practice 4**

*Your model from TP_01 is served by the API from TP_02, and TP_03 keeps an eye on the data it receives. One question is left: **why** does it predict what it predicts? Which features drive its answers, does it treat some groups differently, and can we trust the explanation we are given?*

*In this practice you'll use [SHAP](https://shap.readthedocs.io/en/latest/index.html) to explain several models, check one of them for bias, and then see how a dishonest model owner can make SHAP tell a reassuring story that isn't true.*

*We start with a small medical dataset to learn the tools. In the exercises you'll apply them to your own TP_01 model.*

![ShapValues](https://shap.readthedocs.io/en/latest/_images/shap_header.png)

---

# Setup

The versions are listed in `TP_04/requirements.txt`. On Colab, pip only adds what is missing.

**If pip upgraded or downgraded a package (e.g. SHAP or scikit-learn), restart the kernel before running the next cell.** Python keeps the old version loaded until then.

In [ ]:
import os

REQUIREMENTS_URL = (
    "https://raw.githubusercontent.com/ramonzaca/MLSecOPs/main/TP_04/requirements.txt"
)
requirements = (
    "requirements.txt" if os.path.isfile("requirements.txt") else REQUIREMENTS_URL
)
%pip install -q -r {requirements}

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.cluster.hierarchy
import shap
from sklearn import linear_model, tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    mean_absolute_error,
    root_mean_squared_error,
)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neural_network import MLPRegressor

# SHAP calls the models with plain NumPy arrays, so scikit-learn complains that the column names are
# missing. That one is harmless: silence only it. Don't silence *all* warnings: some of them
# (e.g. a scikit-learn version mismatch when loading a model) matter.
warnings.filterwarnings("ignore", message="X does not have valid feature names")

shap.initjs()  # needed for the interactive force plots

> **SHAP in three ideas**
>
> 1. **Start from the average.** Before looking at a patient, the best guess is the model's average prediction. SHAP calls it the **base value**.
> 2. **Each feature pushes.** For one prediction, each feature gets a **SHAP value**: how much it pushed the prediction up (positive) or down (negative) compared with the average.
> 3. **The pushes add up.** Base value + all SHAP values = the model's prediction, exactly. Nothing is left unexplained.
>
> One prediction explained this way is a **local** explanation. Averaging the size of the pushes over many predictions gives a **global** view: which features matter most to the model overall.

---

# The data

The [Diabetes dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset) describes 442 patients, all diabetic, with 10 measurements (age, sex, body-mass index `bmi`, blood pressure `bp` and six blood tests `s1`–`s6`). The target is a measure of how much the disease progressed one year later: the higher, the worse. All features are already centred and scaled, which is why their values are small and close to 0.

In [ ]:
X, y = shap.datasets.diabetes()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(f"{len(X_train)} training patients, {len(X_test)} test patients")

# Rather than use the whole training set as the background (see the box below), we summarise it
# with 10 weighted k-means centroids, each weighted by the number of patients it represents.
X_train_summary = shap.kmeans(X_train, 10)


def print_rmse(model):
    rmse = root_mean_squared_error(y_test, model.predict(X_test))
    print(f"Root mean squared error on the test set: {rmse:.1f}")
    return rmse


X.head()

> **Why does SHAP need background data?** To measure how much a feature pushes a prediction, SHAP compares the model's answer *with* the feature's real value and *without* it. A model can't run with a missing value, so "without" means "replaced by typical values", taken from a **background** dataset. The bigger the background, the more precise and the slower. A summary (k-means centroids, or a random sample) is the usual compromise.
>
> Keep this in mind for section 7: to explain a model, SHAP feeds it **artificial patients**, part real patient and part background.

---

# 1. Features that travel together

Before explaining any model, let's look at which features are **correlated**, grouped by a hierarchical clustering. Features that move together (for example `s1` and `s2`, two cholesterol measurements) carry the same information. A model can use either one, and the credit can be split between them in ways that look arbitrary.

In [ ]:
partition_tree = shap.utils.partition_tree(X)
plt.figure(figsize=(15, 6))
scipy.cluster.hierarchy.dendrogram(partition_tree, labels=X.columns)
plt.title("Hierarchical clustering of the features")
plt.xlabel("feature")
plt.ylabel("distance")
plt.show()

---

# 2. A linear model, explained

*Let's begin with a basic model, a linear regression.*

In [ ]:
lin_regr = linear_model.LinearRegression()
lin_regr.fit(X_train, y_train)
rmse = {"linear": print_rmse(lin_regr)}

`KernelExplainer` works with **any** model: it only needs a function that makes predictions, and the background data. The price is speed: it calls the model many times per explanation.

In [ ]:
ex = shap.KernelExplainer(lin_regr.predict, X_train_summary)
shap_linear = ex(
    X_test, silent=True
)  # an Explanation object: values, base values, data

*Explain a single prediction: the first patient of the test set.*

In [ ]:
shap.plots.waterfall(shap_linear[0])

In [ ]:
# The three ideas, checked: base value + SHAP values = prediction
patient = X_test.iloc[[0]]
print("base value      :", round(float(shap_linear.base_values[0]), 2))
print("+ SHAP values   :", round(float(shap_linear.values[0].sum()), 2))
print(
    "= explanation   :",
    round(float(shap_linear.base_values[0] + shap_linear.values[0].sum()), 2),
)
print("model prediction:", round(float(lin_regr.predict(patient)[0]), 2))

*The same explanation as an interactive force plot. Red features push the prediction up, blue ones push it down.*

In [ ]:
shap.plots.force(shap_linear[0])

*Now all the predictions of the test set at once. Each dot is one patient. Its position is the SHAP value, and its colour is the feature's value (red = high).*

In [ ]:
shap.plots.beeswarm(shap_linear)

In [ ]:
# How does the effect of bmi change with bmi itself? The colour shows the feature that interacts most with it.
shap.plots.scatter(shap_linear[:, "bmi"], color=shap_linear)

For a linear model the scatter plot is a straight line: the SHAP value of `bmi` is just its coefficient × (bmi − average bmi). That's a good sanity check that the explainer works.

---

# 3. Other models, other stories

*Let's explain three more models on the same data: a decision tree, a random forest and a small neural network.*

For models made of trees, `TreeExplainer` computes **exact** SHAP values, fast, using the structure of the trees. The neural network goes back to `KernelExplainer`.

In [ ]:
dtree = tree.DecisionTreeRegressor(min_samples_split=20, random_state=0)
dtree.fit(X_train, y_train)
rmse["decision tree"] = print_rmse(dtree)
shap_tree = shap.TreeExplainer(dtree)(X_test)
shap.plots.beeswarm(shap_tree)

In [ ]:
rforest = RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1)
rforest.fit(X_train, y_train)
rmse["random forest"] = print_rmse(rforest)
shap_forest = shap.TreeExplainer(rforest)(X_test)
shap.plots.beeswarm(shap_forest)

In [ ]:
shap.plots.scatter(shap_forest[:, "bmi"], color=shap_forest)

In [ ]:
nn = MLPRegressor(
    solver="lbfgs", alpha=1e-1, hidden_layer_sizes=(5, 2), max_iter=1000, random_state=0
)
nn.fit(X_train, y_train)
rmse["neural network"] = print_rmse(nn)
shap_nn = shap.KernelExplainer(nn.predict, X_train_summary)(X_test, silent=True)
shap.plots.beeswarm(shap_nn)

*All the test predictions of the forest in one interactive plot (hover it, and try the drop-downs).*

In [ ]:
shap.plots.force(shap_forest)

*Side by side: the error of each model, and the 3 features that matter most to it (mean |SHAP value|).*

In [ ]:
def top_features(explanation, n=3):
    importance = pd.Series(
        np.abs(explanation.values).mean(axis=0), index=explanation.feature_names
    )
    return ", ".join(importance.sort_values(ascending=False).index[:n])


explanations = {
    "linear": shap_linear,
    "decision tree": shap_tree,
    "random forest": shap_forest,
    "neural network": shap_nn,
}
pd.DataFrame(
    {
        "test RMSE": {name: round(value, 1) for name, value in rmse.items()},
        "top 3 features": {name: top_features(e) for name, e in explanations.items()},
    }
)

> **An explanation explains the model, not the world.** All four models have `s5` in their top 3, but beyond that they disagree: the linear model and the network put `s1` first, and the tree-based models don't have it in their top 3. `s1` travels with `s2` (section 1), and each model spreads the credit between correlated features its own way. SHAP faithfully describes what each model does, including a bad model's quirks (the decision tree is the least accurate). Before trusting an explanation, check that the model is worth explaining.

---

# 4. Choosing an explainer

*How close are the faster, approximate explainers to the exact values?* We explain one patient with the random forest in three ways:

- **Tree SHAP**: exact, but only for tree models.
- **Partition SHAP with the feature clustering from section 1**: it only needs the model's `predict` function, so it works with any model. It treats correlated features as groups, which gives a good estimate with few model calls.
- **Raw Partition SHAP**: the same, without our clustering.

In [ ]:
instance = X_test.iloc[[0]]
background = shap.sample(X_train, 100, random_state=0)

masker = shap.maskers.Partition(background, clustering=partition_tree)
results = {}
for name, explainer in [
    ("Tree SHAP", shap.TreeExplainer(rforest, background)),
    ("Partition SHAP", shap.PartitionExplainer(rforest.predict, masker)),
    ("Raw Partition SHAP", shap.PartitionExplainer(rforest.predict, background)),
]:
    start = time.perf_counter()
    results[name] = explainer(instance)
    print(f"{name:18s}: {time.perf_counter() - start:5.2f} s")

In [ ]:
plt.figure(figsize=(15, 6))
for (name, values), style in zip(results.items(), ["-", "g--", "r--"]):
    plt.plot(X.columns, values[0].values, style, label=name)
plt.ylabel("SHAP value")
plt.legend()
plt.show()

The partition estimates follow the exact values closely. Tree SHAP is by far the fastest because it reads the trees directly, but it only exists for tree models. On large models and datasets, choosing a fast explainer decides whether you can explain every prediction or only a few.

In [ ]:
shap.plots.waterfall(results["Partition SHAP"][0])

---

# 5. Explaining a classifier

*So far the models predicted a number. Now a yes/no question: will the disease progress more than average (class 1) or not (class 0)?*

*(For teaching only. It's not how you'd build a real medical classifier.)*

In [ ]:
threshold = y_train.mean()
y_train_classification = (y_train > threshold).astype(int)
y_test_classification = (y_test > threshold).astype(int)

rf_clf = RandomForestClassifier(max_features=2, n_estimators=50, random_state=0)
rf_clf.fit(X_train, y_train_classification)
y_pred = rf_clf.predict(X_test)

# The true labels come first: classification_report(y_true, y_pred)
print(classification_report(y_test_classification, y_pred))

In [ ]:
# A classifier has one output per class: the explanation has an extra dimension (patients × features × classes)
shap_clf = shap.TreeExplainer(rf_clf)(X_test)
print(shap_clf.shape)
shap.plots.beeswarm(shap_clf[:, :, 1])  # class 1: progression above average

In [ ]:
# The same patient, explained for class 0 and for class 1. Notice something?
pd.DataFrame(
    {"class 0": shap_clf[0, :, 0].values, "class 1": shap_clf[0, :, 1].values},
    index=X.columns,
).round(3)

The two columns are mirror images. The model outputs two probabilities that always add up to 1, so whatever pushes one up pushes the other down by the same amount. With two classes, explaining one of them is enough.

---

# 6. 🔎 Is the model biased?

The dataset contains `sex`, coded as two values (−0.045 and 0.051 after scaling; the documentation doesn't say which is which, so we'll call them **group A** and **group B**). Should a model predicting disease progression depend on sex? Maybe: there are real medical differences. Maybe not: it could also have learned something unfair from the data. Either way, **we must know**.

Two different questions:

1. **Does the model *use* the attribute?** SHAP answers it.
2. **Do the model's *outcomes* differ between groups?** Comparing predictions and errors group by group answers it.

In [ ]:
group = np.where(X_test["sex"] > 0, "B", "A")

importance = pd.Series(
    np.abs(shap_forest.values).mean(axis=0), index=X.columns
).sort_values(ascending=False)
print(
    f"Rank of `sex` among the forest's {len(importance)} features: {list(importance.index).index('sex') + 1}"
)
print(
    f"Average |SHAP value| of `sex`: {importance['sex']:.1f} (top feature `{importance.index[0]}`: {importance.iloc[0]:.1f})"
)

shap.plots.scatter(shap_forest[:, "sex"])

In [ ]:
# Question 2: outcomes, group by group
forest_pred = rforest.predict(X_test)
pd.DataFrame(
    {
        "group": group,
        "actual": y_test,
        "predicted": forest_pred,
        "abs. error": np.abs(y_test - forest_pred),
    }
).groupby("group").agg(
    patients=("actual", "size"),
    mean_actual=("actual", "mean"),
    mean_predicted=("predicted", "mean"),
    MAE=("abs. error", "mean"),
).round(1)

The two questions don't give the same answer. `sex` is one of the features the forest uses least. Yet the gap between the groups' mean predictions is larger than the gap between their actual values, and the model is less accurate for one group than for the other. With 35 and 54 test patients, part of this can be noise (you'd need more data, or confidence intervals, to conclude). But it shows that a small SHAP value for `sex` does **not** prove that the groups are treated alike. The groups also differ in their other features, and those can carry the information:

In [ ]:
# Can the *other* features tell group A from group B?
others = X.drop(columns="sex")
accuracy = cross_val_score(LogisticRegression(), others, X["sex"] > 0, cv=5).mean()
print(
    f"Sex predicted from the other 9 features: {accuracy:.0%} accuracy (a coin flip would be ~{max((X['sex'] > 0).mean(), (X['sex'] <= 0).mean()):.0%})"
)
print(X.corr()["sex"].drop("sex").round(2).sort_values().to_string())

> **Proxies.** `s3` and `s4` are correlated with sex, and together the other features predict it far better than chance. Removing a sensitive attribute from the data doesn't remove the information: a model can rebuild it from **proxies**. In credit scoring, the classic proxy is the postcode (you'll meet it in exercise 2). That's why a bias check always includes question 2, outcomes per group, and not only "is the column in the model?".
>
> Deciding whether a gap is *unfair* isn't a statistics question: it depends on the use of the model and on the law (the EU AI Act treats many uses in health, credit or hiring as high-risk). The data scientist's job is to measure and report it.

---

# 7. 🔐 Scenario: explanations can be fooled

A hospital group uses a model to decide which diabetic patients get a costly follow-up programme. An auditor suspects the model is biased against one group and asks for SHAP explanations of its predictions on the group's patients.

The model owner has something to hide: the model in production **adds 60 points of predicted progression to every patient of group B**. Can they make SHAP say that `sex` doesn't matter, *without changing a single real prediction*?

The idea comes from [Slack et al., *Fooling LIME and SHAP* (2020)](https://arxiv.org/abs/1911.02508). As we saw in the setup box, SHAP explains a model by feeding it **artificial patients** that mix a real patient with background data. If the model can tell real patients from these mixtures, it can behave one way for the real ones and another way for the explainer.

In [ ]:
# An honest model that never looks at sex...
honest = RandomForestRegressor(
    n_estimators=200, min_samples_leaf=5, random_state=0, n_jobs=-1
)
honest.fit(X_train.drop(columns="sex"), y_train)
columns = list(X.columns)


def innocent_model(Z):
    Z = pd.DataFrame(np.asarray(Z), columns=columns)
    return honest.predict(Z.drop(columns="sex"))


# ... and the one really used in production: the same, plus a penalty for group B
def biased_model(Z):
    Z = pd.DataFrame(np.asarray(Z), columns=columns)
    return innocent_model(Z) + 60 * (Z["sex"] > 0).to_numpy()


# The trick: real patients are in the hospital's records. SHAP's artificial mixtures are not.
known_patients = {tuple(row) for row in X.round(8).to_numpy()}


def looks_like_a_real_patient(Z):
    return np.array(
        [
            tuple(row) in known_patients
            for row in np.round(np.asarray(Z, dtype=float), 8)
        ]
    )


def scaffolded_model(Z):
    return np.where(looks_like_a_real_patient(Z), biased_model(Z), innocent_model(Z))


# Every real prediction is the biased one: from the patients' side, nothing has changed
assert np.allclose(scaffolded_model(X_test), biased_model(X_test))
print("Scaffolded model == biased model on all", len(X_test), "test patients")

In [ ]:
# What the auditor sees: SHAP explanations of both models on the same patients
fooled = {}
for name, model in [
    ("biased model", biased_model),
    ("scaffolded model", scaffolded_model),
]:
    explanation = shap.KernelExplainer(model, X_train_summary)(X_test, silent=True)
    importance = pd.Series(
        np.abs(explanation.values).mean(axis=0), index=columns
    ).sort_values(ascending=False)
    fooled[name] = {
        "rank of sex": list(importance.index).index("sex") + 1,
        "mean |SHAP| of sex": round(importance["sex"], 1),
        "top 3 features": ", ".join(importance.index[:3]),
    }
    shap.plots.bar(explanation, show=False)
    plt.title(f"What SHAP says about the {name}")
    plt.show()

fooled = pd.DataFrame(fooled).T
assert (
    fooled.loc["biased model", "rank of sex"] == 1
    and fooled.loc["scaffolded model", "rank of sex"] >= 8
)
fooled

Same predictions, opposite explanations. For the biased model, `sex` is the most important feature. For the scaffolded one, it's among the least important: the explainer's artificial patients all went to the innocent model.

But the section 6 check, outcomes per group, doesn't depend on the explainer:

In [ ]:
pd.DataFrame({"group": group, "predicted": scaffolded_model(X_test)}).groupby("group")[
    "predicted"
].mean().round(1)

> **🔐 What this means for security**
>
> - **An explanation produced by the party you're auditing is a claim, not proof.** A post-hoc explanation only shows how the model behaves on the inputs the explainer chose, and a model can recognise those inputs.
> - Our detector is the crudest possible one: a lookup in the patient records. Slack et al. train a classifier to spot SHAP's and LIME's artificial inputs, which also works on **new** patients.
> - **Defences:** audit **outcomes** on real data (section 6), ideally data the model owner didn't choose; get access to the model itself, not only to its explanations; and check that the explained predictions match the ones actually served (the model file's hash from TP_01 and TP_02 helps here).

---

# Exercises

### Exercise 1: Explain your own model

Replace the dataset and the model with the ones from TP_01 (the California housing data, and the model you exported as `TP_01_model.skops`). Upload the model file next to this notebook (on Colab: the *Files* panel on the left).

The cell below does the plumbing: it downloads the housing data (checking its hash), rebuilds TP_01's test set (same split, same seed), loads your model safely, and turns the test set into the features the final estimator actually sees (`X_housing`, with readable column names). Then it's your turn:

- Which explainer suits your model? (Hint: look at `housing_estimator`. The TP_01 notebook exports a `LinearRegression`. If you exported another model, the answer may differ.)
- Plot the global view (beeswarm or bar), and explain one district with a waterfall plot.

If loading fails with an error such as `AttributeError: 'SimpleImputer' object has no attribute ...`, your model was saved with another scikit-learn version. Re-export it from TP_01 in the same environment as this notebook. (That's the version check TP_02 v3 performs at startup.)

In [ ]:
# --- Exercise 1 plumbing: TP_01 data and model -------------------------------------------
import hashlib
import tarfile
import urllib.request

import skops.io as sio
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedShuffleSplit

HOUSING_URL = (
    "https://github.com/ramonzaca/MLSecOPs/raw/main/datasets/housing/housing.tgz"
)
HOUSING_SHA256 = "d4cd501af90475f09b814c7447c7701f59bf28e8cf1180205ae5ace9737a0109"
MODEL_FILE = "TP_01_model.skops"


def sha256sum(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


os.makedirs("datasets/housing", exist_ok=True)
if not os.path.isfile("datasets/housing/housing.csv"):
    urllib.request.urlretrieve(HOUSING_URL, "datasets/housing/housing.tgz")
    if sha256sum("datasets/housing/housing.tgz") != HOUSING_SHA256:
        raise ValueError("housing.tgz checksum mismatch. Refusing to use it.")
    with tarfile.open("datasets/housing/housing.tgz") as tgz:
        tgz.extractall(path="datasets/housing", filter="data")
housing = pd.read_csv("datasets/housing/housing.csv")

# Same stratified split and seed as TP_01 → same test set, never used for training
income_cat = pd.cut(
    housing["median_income"],
    bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5],
)
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_index, test_index = next(split.split(housing, income_cat))
housing_train, housing_test = housing.iloc[train_index], housing.iloc[test_index]


# The model refers to this class by name (see TP_01 and TP_02): it must exist before loading
rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6


class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room=True):
        self.add_bedrooms_per_room = add_bedrooms_per_room

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
        population_per_household = X[:, population_ix] / X[:, households_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[
                X, rooms_per_household, population_per_household, bedrooms_per_room
            ]
        return np.c_[X, rooms_per_household, population_per_household]

    def get_feature_names_out(self, input_features):
        extra = ["rooms_per_household", "population_per_household"]
        if self.add_bedrooms_per_room:
            extra.append("bedrooms_per_room")
        return np.asarray(list(input_features) + extra, dtype=object)


if not os.path.isfile(MODEL_FILE):
    raise FileNotFoundError(
        f"Upload {MODEL_FILE} (exported at the end of TP_01) next to this notebook."
    )
print(
    "SHA-256 of the model:",
    sha256sum(MODEL_FILE),
    "→ compare it with the one printed in TP_01",
)
untrusted = sio.get_untrusted_types(file=MODEL_FILE)
assert set(untrusted) <= {"__main__.CombinedAttributesAdder", "numpy.dtype"}, (
    f"unexpected types: {untrusted}"
)
housing_model = sio.load(MODEL_FILE, trusted=untrusted)

# SHAP explains the final estimator on the features it receives, after TP_01's preparation pipeline.
# The estimator was trained on a NumPy array: give it `.to_numpy()` when you call it directly.
housing_features = [col for col in housing.columns if col != "median_house_value"]
housing_preparation, housing_estimator = housing_model[:-1], housing_model[-1]
feature_names = housing_preparation.get_feature_names_out()


def prepare(df):
    return pd.DataFrame(
        housing_preparation.transform(df[housing_features]),
        columns=feature_names,
        index=df.index,
    )


X_housing_background = prepare(housing_train.sample(100, random_state=42))
X_housing = prepare(housing_test)
print(
    f"{len(X_housing)} test districts, MAE of your model on them:",
    round(
        mean_absolute_error(
            housing_test["median_house_value"],
            housing_estimator.predict(X_housing.to_numpy()),
        )
    ),
)
print("Final estimator:", housing_estimator)

### Exercise 2: Is your model biased by location?

`ocean_proximity` isn't a protected attribute, but **location often is a proxy for one**. In the US, refusing loans by neighbourhood ("redlining") was a way to discriminate by race without ever writing it down.

- How much does `ocean_proximity` weigh in your model's predictions? SHAP values add up, so the contribution of `ocean_proximity` is the **sum** of its one-hot columns (`cat__ocean_proximity_...`).
- Compare, per `ocean_proximity` category of `housing_test`: the number of districts, the mean actual price, the mean predicted price, and the mean error. Is the model equally accurate everywhere?
- If you removed `ocean_proximity` from the model, would location disappear from its predictions? (Hint: section 6.)

### Exercise 3: Final report

Write a short report (in a markdown cell) with:

- the 3 most important features of your model, and why you think they matter. Do they make sense for house prices? Look at the sign of their effect, and at features that move together (section 1): a linear model can give large, opposite weights to two correlated features;
- what you found in exercise 2;
- what you would tell a regulator who asks you to prove that your model isn't biased, now that you've seen section 7.

### Exercise 4 (optional): The attacker's view

Could the trick of section 7 hide `ocean_proximity` from an auditor who explains your housing model?

- Adapt `scaffolded_model` and check it.
- The lookup in `known_patients` only works on records the model owner already has. What happens if the auditor explains **new** districts? How would you build a detector that works on them too? (Hint: what do SHAP's artificial districts look like? See section 1.)